In [0]:
dbutils.widgets.text("catalog", "dev")
catalog = dbutils.widgets.get("catalog")

In [0]:
# ============================================================
# Data Quality Gate – stops the pipeline on critical failures
# ============================================================
import json
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# --- Config -------------------------------------------------
catalog = dbutils.widgets.get("catalog")

FAILURES = []          # list of critical failures
WARNINGS = []          # list of non-critical warnings
REJECT_REPORT = []     # rejected-row counts per table

CRITICAL_TOL = 0.0001  # 0.01% reconciliation tolerance

# The eight source tables and their known primary keys
SOURCE_TABLES = {
    "booking":     {"bronze": True,  "silver": True},
    "passenger":   {"bronze": True,  "silver": True},
    "route":       {"bronze": True,  "silver": True},
    "airline":     {"bronze": True,  "silver": True},
    "nation":      {"bronze": True,  "silver": True},
    "region":      {"bronze": True,  "silver": True},
    "airline_route":    {"bronze": True,  "silver": True},
    "flight_booking":      {"bronze": True,  "silver": True},
}

# Silver primary-key hints (auto-detect from schema when possible)
SILVER_PK_HINTS = {
    "booking":    ["bk_bookingid", "bookingid"],
    "passenger":  ["psgr_passengerid", "passengerid"],
    "route":      ["rt_routeid", "routeid"],
    "airline":    ["al_airlineid", "airlineid"],
    "nation":     ["nt_nationid", "nationid"],
    "region":     ["rg_regionid", "regionid"],
    "airline_route":  ["airlinerouteid", "ar_airlineid", "ar_routeid"],
    "flight_booking": ["flightid", "fb_bookingid", "fb_linenumber"],
}

# --- Helpers ------------------------------------------------
def table_exists(catalog_name, schema_name, table_name):
    """Return True if a three-part table exists in the metastore."""
    return spark.catalog.tableExists(f"{catalog_name}.{schema_name}.{table_name}")

def add_failure(check_name, message):
    FAILURES.append({"check": check_name, "message": message})

def add_warning(check_name, message):
    WARNINGS.append({"check": check_name, "message": message})

def resolve_pk(df_columns, table_name):
    """Best-effort PK resolution from column names + hint list.
    Returns a *list* of column names to support composite primary keys."""
    hints = SILVER_PK_HINTS.get(table_name, [])
    lower_cols = {c.lower(): c for c in df_columns}
    matched = [lower_cols[h.lower()] for h in hints if h.lower() in lower_cols]
    if matched:
        return matched
    # heuristic: first column ending in 'id'
    for c in df_columns:
        if c.lower().endswith("id") and "passenger" not in c.lower():
            return [c]
    return [df_columns[0]] if df_columns else []

# ============================================================
# 1. Bronze ingestion check – all 8 source tables present
# ============================================================
print("=" * 70)
print("CHECK 1 — Bronze table existence (all 8 sources)")
print("=" * 70)
for tbl in SOURCE_TABLES:
    full = f"{catalog}.bronze.{tbl}"
    exists = table_exists(catalog, "bronze", tbl)
    status = "PASS" if exists else "FAIL"
    print(f"  {full:50s}  {status}")
    if not exists:
        add_failure("bronze_ingestion", f"Missing bronze table: {full}")

# ============================================================
# 2. Primary keys populated and unique (silver layer)
# ============================================================
print("\n" + "=" * 70)
print("CHECK 2 — Primary keys populated & unique (silver)")
print("=" * 70)
for tbl, info in SOURCE_TABLES.items():
    if not table_exists(catalog, "silver", tbl):
        add_warning("silver_exists", f"Silver table {catalog}.silver.{tbl} not found — skipping PK check")
        continue
    df = spark.read.table(f"{catalog}.silver.{tbl}")
    cols = [f.name for f in df.schema.fields]
    pk_cols = resolve_pk(cols, tbl)
    if not pk_cols:
        add_failure("pk_resolution", f"Could not resolve PK for {tbl}")
        continue
    pk_label = ", ".join(pk_cols)

    total_rows   = df.count()
    null_pk_rows = df.filter(" OR ".join(f"`{c}` IS NULL" for c in pk_cols)).count()
    distinct_pk  = df.select(*pk_cols).distinct().count()
    dup_pk_rows  = total_rows - distinct_pk

    # Record rejection counts
    rejected = null_pk_rows + dup_pk_rows
    REJECT_REPORT.append({
        "table": f"{catalog}.silver.{tbl}",
        "check": "primary_key",
        "total_rows": total_rows,
        "null_pk": null_pk_rows,
        "duplicate_pk": dup_pk_rows,
        "rejected_rows": rejected
    })

    pk_ok = (null_pk_rows == 0 and dup_pk_rows == 0)
    status = "PASS" if pk_ok else "FAIL"
    print(f"  {tbl:15s}  pk={pk_label:25s}  nulls={null_pk_rows:>8,}  dups={dup_pk_rows:>8,}  {status}")
    if null_pk_rows > 0:
        add_failure("pk_not_null", f"{tbl} PK ({pk_label}) has {null_pk_rows} NULL values")
    if dup_pk_rows > 0:
        add_failure("pk_not_unique", f"{tbl} PK ({pk_label}) has {dup_pk_rows} duplicate key rows")

# ============================================================
# 3. Gold fact_bookings — no null surrogate keys
# ============================================================
print("\n" + "=" * 70)
print("CHECK 3 — Fact table: no NULL surrogate keys")
print("=" * 70)
if table_exists(catalog, "gold", "fact_bookings"):
    fact = spark.read.table(f"{catalog}.gold.fact_bookings")
    fact_total = fact.count()
    null_sk = fact.filter(F.col("booking_segment_id").isNull()).count()
    print(f"  fact_bookings.booking_segment_id  total={fact_total:,}  nulls={null_sk:,}")
    REJECT_REPORT.append({
        "table": f"{catalog}.gold.fact_bookings",
        "check": "surrogate_key_not_null",
        "total_rows": fact_total,
        "null_sk": null_sk,
        "rejected_rows": null_sk
    })
    if null_sk > 0:
        add_failure("fact_null_sk", f"fact_bookings has {null_sk} rows with NULL booking_segment_id")
    else:
        print("  PASS")
else:
    add_failure("fact_missing", f"Gold table {catalog}.gold.fact_bookings does not exist")

# ============================================================
# 4. Numeric measures — business rules
#    ticket price > 0, discount in [0,1], seats >= 1
# ============================================================
print("\n" + "=" * 70)
print("CHECK 4 — Numeric measures business rules")
print("=" * 70)

# 4a. Silver booking: ticket price > 0
if table_exists(catalog, "silver", "booking"):
    bk = spark.read.table(f"{catalog}.silver.booking")
    bk_cols = [f.name.lower() for f in bk.schema.fields]
    price_col = next((c for c in ["bk_totalprice", "totalprice", "price"] if c.lower() in bk_cols), None)
    if price_col:
        orig_col = next(f.name for f in bk.schema.fields if f.name.lower() == price_col)
        violations_price = bk.filter((F.col(orig_col).isNull()) | (F.col(orig_col) <= 0.0)).count()
        print(f"  silver.booking  {orig_col} > 0.0          violations={violations_price:,}")
        REJECT_REPORT.append({
            "table": f"{catalog}.silver.booking",
            "check": "ticket_price_gt_0",
            "violations": violations_price,
            "rejected_rows": violations_price
        })
        if violations_price > 0.0:
            add_failure("measure_price_gt_0", f"{violations_price} booking rows violate ticket_price > 0")
    else:
        add_warning("measure_price_gt_0", "Could not find a price column in silver.booking")
else:
    add_failure("silver_booking_missing", "silver.booking not found — cannot check ticket price")

# 4b. Discount in [0,1] — check any column that looks like discount
if table_exists(catalog, "silver", "booking"):
    disc_col = next((f.name for f in bk.schema.fields if "discount" in f.name.lower()), None)
    if disc_col:
        violations_disc = bk.filter(
            (F.col(disc_col).isNull()) | (F.col(disc_col) < 0) | (F.col(disc_col) > 1)
        ).count()
        print(f"  silver.booking  {disc_col} in [0,1]    violations={violations_disc:,}")
        REJECT_REPORT.append({
            "table": f"{catalog}.silver.booking",
            "check": "discount_range_0_1",
            "violations": violations_disc,
            "rejected_rows": violations_disc
        })
        if violations_disc > 0:
            add_failure("measure_discount_range", f"{violations_disc} booking rows violate discount in [0,1]")
    else:
        add_warning("measure_discount_range", "No discount column found in silver.booking")

# 4c. Seats >= 1 — check flight or aircraft table for a seats column
seat_check_done = False
for seat_tbl in ["flight_booking", "airline_route"]:
    if table_exists(catalog, "silver", seat_tbl):
        df_seat = spark.read.table(f"{catalog}.silver.{seat_tbl}")
        seat_col = next((f.name for f in df_seat.schema.fields if "seat" in f.name.lower()), None)
        if seat_col:
            violations_seats = df_seat.filter(
                (F.col(seat_col).isNull()) | (F.col(seat_col) < 1)
            ).count()
            print(f"  silver.{seat_tbl:10s}  {seat_col} >= 1        violations={violations_seats:,}")
            REJECT_REPORT.append({
                "table": f"{catalog}.silver.{seat_tbl}",
                "check": "seats_gte_1",
                "violations": violations_seats,
                "rejected_rows": violations_seats
            })
            if violations_seats > 0:
                add_failure("measure_seats_gte_1", f"{violations_seats} {seat_tbl} rows violate seats >= 1")
            seat_check_done = True
            break
if not seat_check_done:
    add_warning("measure_seats_gte_1", "No seats column found in flight or aircraft tables")

# ============================================================
# 5. Gold vs Silver reconciliation (tolerance <= 0.01%)
# ============================================================
print("\n" + "=" * 70)
print("CHECK 5 — Gold vs Silver reconciliation (tol <= 0.01%)")
print("=" * 70)

# 5a. Row-count reconciliation for gold dimensions vs silver sources
DIM_MAP = {
    "dim_passenger": "passenger",
    "dim_route":     "route",
    "dim_airline":   "airline",
}

for gold_tbl, silver_tbl in DIM_MAP.items():
    if not (table_exists(catalog, "gold", gold_tbl) and table_exists(catalog, "silver", silver_tbl)):
        add_warning("recon_dim", f"Skipping {gold_tbl} vs {silver_tbl} — one or both missing")
        continue
    silver_cnt = spark.read.table(f"{catalog}.silver.{silver_tbl}").count()
    gold_cnt  = spark.read.table(f"{catalog}.gold.{gold_tbl}").count()
    if silver_cnt == 0:
        diff_pct = 0.0
    else:
        diff_pct = abs(gold_cnt - silver_cnt) / silver_cnt
    status = "PASS" if diff_pct <= CRITICAL_TOL else "FAIL"
    print(f"  rows  silver.{silver_tbl:12s}={silver_cnt:>10,}  gold.{gold_tbl:15s}={gold_cnt:>10,}  diff%={diff_pct:.6f}  {status}")
    if diff_pct > CRITICAL_TOL:
        add_failure("recon_rows", f"{gold_tbl} rows ({gold_cnt}) vs silver.{silver_tbl} ({silver_cnt}) differ by {diff_pct:.4%} > {CRITICAL_TOL:.4%}")

# 5b. Revenue reconciliation: gold view vs silver booking
if table_exists(catalog, "gold", "vw_annual_booking_kpis") and table_exists(catalog, "silver", "booking"):
    gold_revenue = spark.read.table(f"{catalog}.gold.vw_annual_booking_kpis").agg(
        F.sum("net_revenue").alias("gold_total")
    ).collect()[0]["gold_total"] or 0.0

    silver_revenue = spark.read.table(f"{catalog}.silver.booking").agg(
        F.sum("bk_totalprice").alias("silver_total")
    ).collect()[0]["silver_total"] or 0.0

    if silver_revenue == 0:
        rev_diff_pct = 0.0
    else:
        rev_diff_pct = abs(gold_revenue - silver_revenue) / abs(silver_revenue)

    status = "PASS" if rev_diff_pct <= CRITICAL_TOL else "FAIL"
    print(f"  rev   silver.booking={silver_revenue:>18,.2f}  gold.vw_kpis={gold_revenue:>18,.2f}  diff%={rev_diff_pct:.6f}  {status}")
    if rev_diff_pct > CRITICAL_TOL:
        add_failure("recon_revenue", f"Gold revenue ({gold_revenue:,.2f}) vs silver ({silver_revenue:,.2f}) differ by {rev_diff_pct:.4%} > {CRITICAL_TOL:.4%}")
else:
    add_warning("recon_revenue", "Cannot reconcile revenue — vw_annual_booking_kpis or silver.booking missing")

# ============================================================
# 6. Rejected record counts per table — summary report
# ============================================================
print("\n" + "=" * 70)
print("CHECK 6 — Rejected record counts per table")
print("=" * 70)
if REJECT_REPORT:
    reject_df = spark.createDataFrame(REJECT_REPORT)
    reject_df.display()
else:
    print("  No rejection metrics recorded.")

# ============================================================
# Final gate — raise pipeline-stopping error on any critical failure
# ============================================================
print("\n" + "=" * 70)
print("DATA QUALITY GATE SUMMARY")
print("=" * 70)
print(f"  Critical failures : {len(FAILURES)}")
print(f"  Warnings         : {len(WARNINGS)}")

if FAILURES:
    print("\n--- CRITICAL FAILURES ---")
    for i, fail in enumerate(FAILURES, 1):
        print(f"  [{i}] {fail['check']}: {fail['message']}")
    if WARNINGS:
        print("\n--- WARNINGS (non-blocking) ---")
        for i, warn in enumerate(WARNINGS, 1):
            print(f"  [{i}] {warn['check']}: {warn['message']}")
    raise RuntimeError(
        f"DATA QUALITY GATE FAILED — {len(FAILURES)} critical check(s) violated. "
        f"Pipeline execution stopped.\n" +
        "\n".join(f"  - {f['check']}: {f['message']}" for f in FAILURES)
    )
else:
    if WARNINGS:
        print("\n--- WARNINGS (non-blocking) ---")
        for i, warn in enumerate(WARNINGS, 1):
            print(f"  [{i}] {warn['check']}: {warn['message']}")
    print("\n✅ All critical data quality checks passed. Pipeline may proceed.")

CHECK 1 — Bronze table existence (all 8 sources)
  dev.bronze.booking                                  PASS
  dev.bronze.passenger                                PASS
  dev.bronze.route                                    PASS
  dev.bronze.airline                                  PASS
  dev.bronze.nation                                   PASS
  dev.bronze.region                                   PASS
  dev.bronze.airline_route                            PASS
  dev.bronze.flight_booking                           PASS

CHECK 2 — Primary keys populated & unique (silver)
  booking          pk=bk_bookingid               nulls=       0  dups=       0  PASS
  passenger        pk=pa_passengerid             nulls=       0  dups=       0  PASS
  route            pk=ro_routeid                 nulls=       0  dups=       0  PASS
  airline          pk=al_airlineid               nulls=       0  dups=       0  PASS
  nation           pk=n_nationkey                nulls=       0  dups=       0  PASS
  

check,duplicate_pk,null_pk,rejected_rows,table,total_rows,null_sk,violations
primary_key,0,0,0,dev.silver.booking,8030,null,null
primary_key,0,0,0,dev.silver.passenger,1500,null,null
primary_key,0,0,0,dev.silver.route,100,null,null
primary_key,0,0,0,dev.silver.airline,10,null,null
primary_key,0,0,0,dev.silver.nation,25,null,null
primary_key,0,0,0,dev.silver.region,5,null,null
primary_key,0,0,0,dev.silver.airline_route,401,null,null
primary_key,0,0,0,dev.silver.flight_booking,25023,null,null
surrogate_key_not_null,null,null,0,dev.gold.fact_bookings,8030,0,null
ticket_price_gt_0,null,null,0,dev.silver.booking,null,null,0



DATA QUALITY GATE SUMMARY
  Critical failures : 0
  Warnings         : 1

--- WARNINGS (non-blocking) ---
  [1] measure_discount_range: No discount column found in silver.booking

✅ All critical data quality checks passed. Pipeline may proceed.
